In [11]:
%pip install python-louvain networkx pandas pyarrow

import networkx as nx
import community as community_louvain  
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from datetime import datetime

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
# Đọc file CSV từ local
import pandas as pd
import time


csv_path = "../data/processed/subreddit_similarity_results.csv"
df_sim = pd.read_csv(csv_path)

# Đổi tên cột cho khớp với code cũ
df_sim.columns = ['Subreddit_A', 'Subreddit_B', 'Similarity_Score']

elapsed = time.time() - start_time
print(f"Tổng số cặp similarity: {len(df_sim)}")

Tổng số cặp similarity: 134866


In [20]:
# Tính threshold (97th percentile) bằng pandas

threshold = df_sim['Similarity_Score'].quantile(0.97)
print(f"Threshold (97th percentile): {threshold:.4f}")

# Lọc các cạnh có similarity >= threshold
df_filtered = df_sim[df_sim['Similarity_Score'] >= threshold]

print(f"Số cạnh sau lọc: {len(df_filtered)}")

Threshold (97th percentile): 0.9942
Số cạnh sau lọc: 4046


In [14]:
# Dữ liệu đã là pandas DataFrame, không cần convert
edges_pd = df_filtered[["Subreddit_A", "Subreddit_B", "Similarity_Score"]]

In [15]:
start_time = time.time()
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Bắt đầu xây dựng graph...")

G = nx.Graph()
for _, row in edges_pd.iterrows():
    G.add_edge(
        row["Subreddit_A"],
        row["Subreddit_B"],
        weight=float(row["Similarity_Score"])
    )

elapsed = time.time() - start_time
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Hoàn thành xây dựng graph trong {elapsed:.2f}s")

[2026-05-10 21:27:35] Bắt đầu xây dựng graph...
Graph: 1037 nodes, 4046 edges
[2026-05-10 21:27:35] Hoàn thành xây dựng graph trong 0.15s


In [16]:
start_time = time.time()
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Bắt đầu phát hiện communities (Louvain algorithm)...")

partition = community_louvain.best_partition(G, weight='weight', resolution=1.0)

num_communities = len(set(partition.values()))
elapsed = time.time() - start_time
print(f"Số communities phát hiện được: {num_communities}")
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Hoàn thành phát hiện communities trong {elapsed:.2f}s")

[2026-05-10 21:27:39] Bắt đầu phát hiện communities (Louvain algorithm)...
Số communities phát hiện được: 149
[2026-05-10 21:27:39] Hoàn thành phát hiện communities trong 0.13s


In [0]:
modularity = community_louvain.modularity(partition, G, weight='weight')
print(f"Modularity score: {modularity:.4f}")

Modularity score: 0.6293


In [17]:
community_df = pd.DataFrame([
    {"subreddit": sub, "community_id": comm_id}
    for sub, comm_id in partition.items()
])

community_sizes = community_df.groupby("community_id").size().reset_index(name="community_size")
community_df = community_df.merge(community_sizes, on="community_id")
community_df = community_df.sort_values("community_size", ascending=False)

print("\nTop 10 communities lớn nhất:")
print(community_df.groupby("community_id")["community_size"].first()
      .sort_values(ascending=False).head(10))


Top 10 communities lớn nhất:
community_id
22    125
9      87
15     84
2      68
29     51
31     49
95     37
0      33
47     31
6      31
Name: community_size, dtype: int64


In [ ]:
# Lưu kết quả ra file CSV local

out_path = "../data/processed/community_results.csv"
community_df.to_csv(out_path, index=False)

elapsed = time.time() - start_time
print(f"Đã lưu community results tại: {out_path}")

[2026-05-10 21:27:49] Bắt đầu lưu kết quả...
Đã lưu community results tại: ../data/processed/community_results.csv
[2026-05-10 21:27:49] Hoàn thành lưu file trong 0.00s
